In [ ]:
import numpy as np


def layer_norm(x, gamma, beta, eps=1e-5):
    """Normalize to mean=0, std=1, then scale and shift"""
    mean = np.mean(x, axis=-1, keepdims=True)
    std = np.std(x, axis=-1, keepdims=True)
    normalized = (x - mean) / (std + eps)
    return gamma * normalized + beta


def softmax(x, axis=-1):
    """Convert scores to probabilities"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def multi_head_attention(x, W_Q, W_K, W_V, W_O, n_heads, mask=None):
    """Multi-head self-attention"""
    seq_len, d_model = x.shape
    d_k = d_model // n_heads

    # Project to Q, K, V
    Q = x @ W_Q
    K = x @ W_K
    V = x @ W_V

    # Reshape for multi-head: (n_heads, seq_len, d_k)
    Q = Q.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    K = K.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    V = V.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)

    # Attention scores
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask

    weights = softmax(scores)
    attended = weights @ V

    # Concatenate heads
    output = attended.transpose(1, 0, 2).reshape(seq_len, d_model)
    return output @ W_O


# Load trained weights
print("Loading model...\n")
weights = np.load("tiny_english_gpt.npz", allow_pickle=True)

print("weights: ", weights)
vocab = weights["vocab"].tolist()


def predict_next(prompt):
    """Predict next word from prompt"""
    # Tokenize
    tokens = [vocab.index(w) for w in prompt.lower().split()]
    seq_len = len(tokens)

    # Get embeddings
    x = weights["token_embed.weight"][tokens]
    x = x + weights["pos_embed.weight"][:seq_len]

    # Causal mask (future tokens can't see each other)
    mask = np.triu(np.ones((seq_len, seq_len)) * -1e9, k=1)

    # Layer norm before attention
    x_norm = layer_norm(
        x, weights["blocks.0.norm1.weight"], weights["blocks.0.norm1.bias"]
    )

    # Multi-head attention
    attn_out = multi_head_attention(
        x_norm,
        weights["blocks.0.attn.W_q.weight"].T,
        weights["blocks.0.attn.W_k.weight"].T,
        weights["blocks.0.attn.W_v.weight"].T,
        weights["blocks.0.attn.W_o.weight"].T,
        n_heads=4,
        mask=mask,
    )

    # Residual connection
    x = x + attn_out

    # Final layer norm
    x = layer_norm(x, weights["ln_f.weight"], weights["ln_f.bias"])

    # Predict from last position
    logits = x[-1] @ weights["lm_head.weight"].T
    probs = softmax(logits)

    # Show top predictions
    top_indices = np.argsort(probs)[-3:][::-1]
    print(f"'{prompt}' → ", end="")
    for idx in top_indices:
        print(f"{vocab[idx]}:{probs[idx]:.0%} ", end="")
    print()


# Test predictions
predict_next("the cat sat on the")
predict_next("the dog ran to the")
predict_next("the big cat sat on the big")

print("\n→ Our model predicts next words with context awareness!")


Loading model...

weights:  NpzFile 'tiny_english_gpt.npz' with keys: token_embed.weight, pos_embed.weight, blocks.0.attn.W_q.weight, blocks.0.attn.W_q.bias, blocks.0.attn.W_k.weight...
[[ 8.00158903e-02  7.44991824e-02  1.99790392e-02 -1.81777894e-01
   1.16029181e-01 -1.20682560e-01  1.26246154e-01 -8.68694335e-02
   8.80061835e-03  5.01867756e-02  3.54446582e-02 -2.93660816e-02
   3.79644394e-01  9.13380265e-01  8.67232680e-02 -1.35786667e-01
  -1.45168781e-01  1.27217203e-01  2.15534177e-02 -4.77515869e-02
  -8.40323567e-02 -2.12831609e-02  2.32838884e-01 -1.03781752e-01
  -1.50481552e-01 -1.53292969e-01  1.92377456e-02  2.60400712e-01
  -5.08804992e-02  8.16716999e-02 -2.17973650e-01 -1.78097948e-01
  -1.22488745e-01  1.04395092e-01  1.15753282e-02 -1.75279111e-01
   6.20807335e-02 -1.01288348e-01 -6.51827097e-01 -1.44145817e-01
   1.41004860e-01  1.22573450e-01  3.07850298e-02  8.25687572e-02
   6.81106821e-02  1.95419773e-01 -9.24823433e-02  1.14924751e-01
   2.09280960e-02  8.1